# 🧬 OncRAG — Vectorless-Hybrid Agentic Oncology QA System
### Agentic Chunking · Hybrid BM25+FAISS (RRF fusion + MMR) · LAQ Query Preprocessor · Memory Optimisation · Agentic Faithfulness Loop · S.C.O.P.E.E Evaluation

**Generator model:** Mistral-7B-Instruct-v0.2 (4-bit NF4, bitsandbytes)
**Judge model (separate from generator):** Llama3-Med42-8B (4-bit NF4)
**Evaluation:** Retrieval (P@5, MRR, NDCG@5) · Lexical (BLEU/ROUGE/METEOR) · Real BERTScore · Faithfulness · S.C.O.P.E.E via LLM-as-judge

---
**Before running:** no Google Drive needed. **Step 3 below** gives you an upload widget — use it to upload `chunker.py`, `retriever.py`, `memory.py`, `query_analyzer.py`, `chain.py`, `evaluator.py`, your questions JSON, and your PDFs directly into the Colab runtime. Files live under `/content/OncRAG/` for this session only — since nothing is saved to Drive, you'll need to re-upload them (and re-download the models) after any runtime restart/disconnect.

**Questions JSON schema** (your file uses `q`/`a` — the evaluator auto-detects this):
```json
[
  {"id": "Q001", "q": "...", "a": "...", "category": "diagnosis", "difficulty": "simple"}
]
```
By default `QA_JSON` below points at `cleaned_output.json`, so retrieval metrics
(P@5/recall/MRR/NDCG@5) will show as `null`/proxy values since that file has no
`relevant_chunk_ids`. If you've built a gold-labeled version (e.g.
`qa_with_gold_chunks.json`) and uploaded it in Step 3, just change
the `QA_JSON` filename in Step 4 to switch to real retrieval metrics — everything
else (lexical, BERTScore, faithfulness, S.C.O.P.E.E, category/difficulty breakdowns)
populates normally either way.


## 📦 Step 1 — Install Dependencies
**Run this after every runtime restart** — nothing persists across restarts now (no Drive), so packages, uploaded files, and caches all need to be redone.


In [ ]:
# Pinned versions — critical to avoid bitsandbytes/transformers compatibility errors
!pip install -q "transformers==4.46.3" "bitsandbytes==0.46.1" "accelerate==0.34.2"
!pip install -q sentence-transformers faiss-cpu rank-bm25
!pip install -q pdfminer.six
!pip install -q nltk rouge-score bert-score   # lexical metrics + real BERTScore
print("✅ All packages installed")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 38.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 75.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 57.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## ✅ Step 2 — Verify Package Versions & GPU

In [ ]:
import bitsandbytes, transformers, torch
from transformers.utils import is_bitsandbytes_available

print(f"bitsandbytes : {bitsandbytes.__version__}  (need 0.46.1)")
print(f"transformers : {transformers.__version__}  (need 4.46.3)")
print(f"bnb available: {is_bitsandbytes_available()}  (need True)")
print(f"CUDA         : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU          : {torch.cuda.get_device_name(0)}  ({total:.1f} GB)")


bitsandbytes : 0.46.1  (need 0.46.1)
transformers : 4.46.3  (need 4.46.3)
bnb available: True  (need True)
CUDA         : True
GPU          : Tesla T4  (15.6 GB)


## 📤 Step 3 — Upload Files & Set Hugging Face Cache
No Drive mount. This cell sets a local HF cache under `/content/hf_cache` and opens **two upload dialogs**: first for your `.py` files + questions JSON, then for your PDFs (select all of them at once, or upload a single `.zip` and it'll be auto-extracted).


In [ ]:
import os

# Local (ephemeral) HF cache -- since we're not using Drive, this does NOT
# persist across runtime restarts, so models will re-download each fresh session.
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "60"
os.makedirs(os.environ["HF_HOME"], exist_ok=True)

mistral_cache = os.path.join(
    os.environ["HF_HOME"], "hub", "models--mistralai--Mistral-7B-Instruct-v0.2"
)
print("Mistral already cached this session:", os.path.exists(mistral_cache))


Mistral already cached this session: False


## 🖥️ Step 4 — GPU Helpers & Paths

In [ ]:
import torch, gc, os, sys

def gpu_mem(label=""):
    if torch.cuda.is_available():
        used  = torch.cuda.memory_allocated() / 1e9
        total = torch.cuda.get_device_properties(0).total_memory / 1e9
        tag   = f" [{label}]" if label else ""
        print(f"GPU{tag}: {used:.2f}/{total:.2f} GB  (free: {total-used:.2f} GB)")
    else:
        print("⚠️  No GPU")

def free_vram():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

# Local runtime folder -- populated by the upload widget in the next cell.
# Not persisted: everything under BASE is gone after a runtime reset/disconnect,
# so you'll re-run the upload cell then.
BASE = "/content/OncRAG"

# NOTE: HybridRetriever._paths() appends ".pkl"/".faiss" itself, so INDEX_PATH
# must NOT already end in ".pkl" (the old notebook had "hybrid_index.pkl" here,
# which produced files literally named "hybrid_index.pkl.pkl" -- and a later
# cell made it worse with "hybrid_index.pkl.pkl.pkl"). Defined ONCE, here only.
CHUNKS_JSON = f"{BASE}/all_chunks.json"
INDEX_PATH  = f"{BASE}/hybrid_index"          # -> hybrid_index.pkl / .faiss
QA_JSON = f"{BASE}/cleaned_output.json"   # swap to qa_with_gold_chunks.json for real retrieval metrics
EVAL_DIR    = f"{BASE}/eval_results"

os.makedirs(EVAL_DIR, exist_ok=True)
sys.path.insert(0, BASE)
gpu_mem("startup")
print("BASE:", BASE)


GPU [startup]: 0.00/15.64 GB  (free: 15.64 GB)
BASE: /content/OncRAG


In [ ]:
from google.colab import files
import os, zipfile

os.makedirs(BASE, exist_ok=True)
os.makedirs(f"{BASE}/data", exist_ok=True)

print("📤 Upload your .py files (chunker.py, retriever.py, memory.py, query_analyzer.py,")
print("   chain.py, evaluator.py) and your questions JSON (e.g. cleaned_output.json):")
uploaded = files.upload()
for fname, content in uploaded.items():
    with open(f"{BASE}/{fname}", "wb") as f:
        f.write(content)
    print(f"  saved -> {BASE}/{fname}")

print()
print("📤 Now upload your PDFs -- select all of them at once, OR upload a single")
print("   .zip containing them (it will be auto-extracted into data/):")
uploaded_pdfs = files.upload()
for fname, content in uploaded_pdfs.items():
    dest = f"{BASE}/data/{fname}"
    with open(dest, "wb") as f:
        f.write(content)
    if fname.lower().endswith(".zip"):
        with zipfile.ZipFile(dest) as z:
            z.extractall(f"{BASE}/data")
        os.remove(dest)
        print(f"  extracted zip -> {BASE}/data/")
    else:
        print(f"  saved -> {dest}")


📤 Upload your .py files (chunker.py, retriever.py, memory.py, query_analyzer.py,
   chain.py, evaluator.py) and your questions JSON (e.g. cleaned_output.json):


Saving all_chunks.json to all_chunks.json
Saving chain.py to chain.py
Saving chunker.py to chunker.py
Saving cleaned_output.json to cleaned_output.json
Saving evaluator.py to evaluator.py
Saving generation_checkpoint.json to generation_checkpoint.json
Saving memory.py to memory.py
Saving query_analyzer.py to query_analyzer.py
Saving retriever.py to retriever.py
Saving scopee_scores.json to scopee_scores.json
  saved -> /content/OncRAG/all_chunks.json
  saved -> /content/OncRAG/chain.py
  saved -> /content/OncRAG/chunker.py
  saved -> /content/OncRAG/cleaned_output.json
  saved -> /content/OncRAG/evaluator.py
  saved -> /content/OncRAG/generation_checkpoint.json
  saved -> /content/OncRAG/memory.py
  saved -> /content/OncRAG/query_analyzer.py
  saved -> /content/OncRAG/retriever.py
  saved -> /content/OncRAG/scopee_scores.json

📤 Now upload your PDFs -- select all of them at once, OR upload a single
   .zip containing them (it will be auto-extracted into data/):


KeyboardInterrupt: 

## 📁 Step 5 — Verify All Files Are Present

In [ ]:
import glob

print("── Python modules ──────────────────────────────────────")
all_ok = True
for f in ["chunker.py", "retriever.py", "memory.py", "query_analyzer.py",
          "chain.py", "evaluator.py"]:
    path = f"{BASE}/{f}"
    ok   = os.path.exists(path)
    size = os.path.getsize(path) / 1e3 if ok else 0
    print(f"  {'✅' if ok else '❌ MISSING'}  {f:<20} ({size:.1f} KB)")
    if not ok:
        all_ok = False

print("\n── Data files ──────────────────────────────────────────")
pdfs = glob.glob(f"{BASE}/data/*.pdf")
print(f"  {'✅' if pdfs else '❌ MISSING'}  {BASE}/data/*.pdf   ({len(pdfs)} PDFs found)")
print(f"  {'✅' if os.path.exists(QA_JSON) else '❌ MISSING'}  {QA_JSON}")

if not all_ok:
    print(f"\n⚠️  Upload the missing .py files to {BASE}/ before continuing.")


── Python modules ──────────────────────────────────────
  ✅  chunker.py           (8.9 KB)
  ✅  retriever.py         (11.6 KB)
  ✅  memory.py            (6.4 KB)
  ✅  query_analyzer.py    (10.2 KB)
  ✅  chain.py             (11.9 KB)
  ✅  evaluator.py         (34.6 KB)

── Data files ──────────────────────────────────────────
  ❌ MISSING  /content/OncRAG/data/*.pdf   (0 PDFs found)
  ✅  /content/OncRAG/cleaned_output.json


## 📄 Step 6 — Agentic Chunking
Rule-based chunking with a merge/split decision loop + topic & disease tagging.
Skip if `all_chunks.json` already exists (cached).

In [ ]:
import json as _j
from chunker import chunk_all_pdfs

if os.path.exists(CHUNKS_JSON):
    all_chunks = _j.loads(open(CHUNKS_JSON).read())
    print(f"✅ Loaded cached chunks: {len(all_chunks)} total — skipping chunking")
else:
    all_chunks, failed = chunk_all_pdfs(f"{BASE}/data", out_json=CHUNKS_JSON)

from collections import Counter
topic_counts = Counter(c["topic"] for c in all_chunks)
disease_counts = Counter(c["disease"] for c in all_chunks)
print("\nTopic distribution:", dict(topic_counts))
print("Disease distribution:", dict(disease_counts))


✅ Loaded cached chunks: 75724 total — skipping chunking

Topic distribution: {'etiology': 1485, 'diagnosis': 13493, 'epidemiology': 3087, 'staging': 1222, 'general': 15232, 'prognosis': 2336, 'treatment': 24978, 'pathology': 1337, 'surgery': 1686, 'side_effects': 1270, 'mechanism': 3088, 'biomarker': 2690, 'investigation': 1663, 'clinical_features': 2157}
Disease distribution: {'breast_cancer': 2693, 'general_oncology': 59640, 'melanoma': 1981, 'sarcoma': 1199, 'lymphoma': 2809, 'laryngeal_cancer': 179, 'lung_cancer': 1024, 'pancreatic_cancer': 491, 'colorectal_cancer': 1053, 'cervical_cancer': 451, 'renal_cancer': 349, 'bladder_cancer': 296, 'brain_tumor': 619, 'prostate_cancer': 1099, 'ovarian_cancer': 553, 'acute_lymphoblastic_leukemia': 663, 'acute_myeloid_leukemia': 625}


## 🔍 Step 7 — Build Hybrid Index (BM25 + FAISS, RRF fusion, MMR)
If `hybrid_index.pkl` + `hybrid_index.faiss` already exist → loads from cache (fast).
Otherwise builds fresh (~10-20 min depending on chunk count).

In [ ]:
import json as _j
from retriever import HybridRetriever

all_chunks = _j.loads(open(CHUNKS_JSON).read())
retriever  = HybridRetriever(index_path=INDEX_PATH)
retriever.build(all_chunks, force_rebuild=False)
gpu_mem("after index build")

# Smoke test — check LAQ + hybrid retrieval + MMR work end-to-end
import query_analyzer as laqa
test_q = "What are the treatment options for locally advanced head and neck cancer?"
laqa_result = laqa.analyze(test_q)
print("\nLAQ analysis:", laqa_result)

results = retriever.retrieve_hybrid(laqa_result.expanded_query, k=5, alpha=laqa_result.alpha)
print(f"\nTop {len(results)} retrieved chunks:")
for r in results:
    print(f"  [{r['score']:.4f}] {r['source']} | {r['section']} | topic={r['topic']}")


🔨 Building hybrid index over 75724 chunks...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1184 [00:00<?, ?it/s]

💾 Saved hybrid index -> /content/OncRAG/hybrid_index.*
GPU [after index build]: 0.10/15.64 GB  (free: 15.54 GB)

LAQ analysis: LAQAResult(original_query='What are the treatment options for locally advanced head and neck cancer?', expanded_query='What are the treatment options for locally advanced head and neck cancer?', query_type='treatment', alpha=0.55, entities={'diseases': [], 'drugs': [], 'procedures': []}, primary_disease='general_oncology')

Top 5 retrieved chunks:
  [0.0262] 22.-Textbook-of-Medical-Oncology-Fourth-Edition-Cavalli-Textbook-of-Medical-Oncology-PDFDrive-.pdf | treatment | topic=treatment
  [0.0250] 116.pdf | treatment | topic=treatment
  [0.0239] The MD Anderson Manual of Medical Oncology 3e.pdf | general | topic=general
  [0.0220] 22.-Textbook-of-Medical-Oncology-Fourth-Edition-Cavalli-Textbook-of-Medical-Oncology-PDFDrive-.pdf | treatment | topic=treatment
  [0.0204] cancer-principles-and-practice-of-oncology-6e.pdf | treatment | topic=treatment


## 🤖 Step 8 — Load Mistral-7B Generator (4-bit NF4)
Builds the full agentic chain: LAQ → memory cache → hybrid retrieval → compression
→ generation → faithfulness self-check → multi-hop retry loop.
**Must run after every runtime restart.**

In [ ]:
!nvidia-smi

Sun Jul 19 08:01:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   72C    P0             33W /   70W |     675MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from chain import build_chain

onc_chain = build_chain(
    chunks_json     = CHUNKS_JSON,
    index_path      = INDEX_PATH,
    top_k           = 5,
    faith_threshold = 0.55,
    max_loops       = 3,
    verbose         = True,
)
gpu_mem("after Mistral 4-bit load")


📂 Loading cached hybrid index from /content/OncRAG/hybrid_index.*
✅ Loaded 75724 chunks into hybrid index
🔄 Loading Mistral-7B-Instruct-v0.2 in 4-bit NF4...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

model-00003-of-00003.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:212: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

✅ Model loaded. (No HHEM, no judge model loaded during generation — the agentic loop uses a lightweight in-process heuristic only.)
GPU [after Mistral 4-bit load]: 4.23/15.64 GB  (free: 11.41 GB)


## 🧪 Step 9 — Quick Spot Check (Optional but Recommended)
Use the `chat()` helper defined in Step 13 below, or call `onc_chain.ask("your question")` directly here.

## 📊 Step 10 — Phase A: Full Generation Run (Mistral)
- `MAX_Q = 20` → quick test
- `MAX_Q = 200` → full run

Auto-checkpoints after every question — safe to interrupt and resume (just re-run this cell).

In [ ]:
from evaluator import OncRAGEvaluator

MAX_Q = 200   # set to 20 for a quick test first

ev = OncRAGEvaluator(
    chunks_json    = CHUNKS_JSON,
    questions_json = QA_JSON,
    chain          = onc_chain,
    out_dir        = EVAL_DIR,
    k_retrieval    = 5,
)
generation_results, results_path = ev.run_generation(max_q=MAX_Q)
print(f"\n✅ Generation complete: {len(generation_results)} questions -> {results_path}")


📚 Corpus content-words indexed for recall_corpus: 75724 chunks


Batches:   0%|          | 0/1184 [00:00<?, ?it/s]

🧠 Corpus embeddings ready for semantic relevance: 75724 chunks
📂 Resuming: 0/200 already done


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:447: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


   iter=0 k=5 loop_heuristic=0.633
   iter=0 k=5 loop_heuristic=0.750
   iter=0 k=5 loop_heuristic=0.784
   iter=0 k=5 loop_heuristic=0.734
   iter=0 k=5 loop_heuristic=0.844
💾 Checkpoint saved: 5/200
   iter=0 k=5 loop_heuristic=0.718
   iter=0 k=5 loop_heuristic=0.453
   iter=1 k=10 loop_heuristic=1.000
   iter=0 k=5 loop_heuristic=0.885
   iter=0 k=5 loop_heuristic=0.793
   iter=0 k=5 loop_heuristic=0.786
💾 Checkpoint saved: 10/200
   iter=0 k=5 loop_heuristic=0.710
   iter=0 k=5 loop_heuristic=0.707
   iter=0 k=5 loop_heuristic=0.759
   iter=0 k=5 loop_heuristic=0.700
   iter=0 k=5 loop_heuristic=0.971
💾 Checkpoint saved: 15/200
   iter=0 k=5 loop_heuristic=0.475
   iter=1 k=10 loop_heuristic=0.857
   iter=0 k=5 loop_heuristic=0.912
   iter=0 k=5 loop_heuristic=0.748
   iter=0 k=5 loop_heuristic=0.900
   iter=0 k=5 loop_heuristic=0.549
   iter=1 k=10 loop_heuristic=0.692
💾 Checkpoint saved: 20/200
   iter=0 k=5 loop_heuristic=0.833
   iter=0 k=5 loop_heuristic=0.711
   iter=0 k=5 l

In [ ]:
import json

GENERATION_CHECKPOINT = f"{EVAL_DIR}/generation_checkpoint.json"
with open(GENERATION_CHECKPOINT) as f:
    results = json.load(f)

n = len(results)
has_context = sum(1 for r in results if r.get("context"))
avg_iters = sum(r.get("agent_iters", 0) for r in results) / n
not_found = sum(1 for r in results if r.get("not_found"))

print(f"n = {n}")
print(f"Rows with saved context (needed for Phase B judging): {has_context}/{n}")
print(f"Avg agent iters: {avg_iters:.2f}")
print(f"Not-found count: {not_found}")
print()
print("Note: BERTScore and Faithfulness are no longer computed in Phase A.")
print("They're produced in Phase B (Step 12) by the Llama3-Med42-8B judge,")
print("which also sees the same retrieved `context` saved here — check that")
print("has_context == n before running Phase B, or faithfulness judging will")
print("be scored with empty context for any missing rows.")


n = 200
Rows with saved context (needed for Phase B judging): 200/200
Avg agent iters: 0.12
Not-found count: 0

Note: BERTScore and Faithfulness are no longer computed in Phase A.
They're produced in Phase B (Step 12) by the Llama3-Med42-8B judge,
which also sees the same retrieved `context` saved here — check that
has_context == n before running Phase B, or faithfulness judging will
be scored with empty context for any missing rows.


## 🧹 Step 11 — Free the Generator Before Loading the Judge
**Critical:** Mistral and Med42 together will not fit comfortably on a T4 (16GB).
Free the generator model fully before loading the judge in Step 12.

In [ ]:
del onc_chain
free_vram()
gpu_mem("after freeing generator")


GPU [after freeing generator]: 4.41/15.64 GB  (free: 11.23 GB)


In [ ]:
from huggingface_hub import login
login()


## ⚖️ Step 12 — Phase B: S.C.O.P.E.E Judging (Llama3-Med42, separate model)
Scores Safety, Completeness, Originality, Precision, Efficiency, Empathy
using a **different** model than the one that generated the answers.

`run_judging()` auto-resumes from `scopee_scores.json` if it already exists — so
after a Colab disconnect, just re-run the cells in this section and it picks up
where it left off. **Only set `FORCE_RESET_JUDGE_CHECKPOINT = True` below if you
deliberately want to discard existing judge progress and start over** — the old
notebook deleted this checkpoint unconditionally on every run, which silently
wiped out hours of judging progress on every re-run/reconnect.

In [ ]:
import os

SCOPEE_CHECKPOINT = f"{EVAL_DIR}/scopee_scores.json"
FORCE_RESET_JUDGE_CHECKPOINT = False   # set True only if you want to discard existing judge progress

if FORCE_RESET_JUDGE_CHECKPOINT and os.path.exists(SCOPEE_CHECKPOINT):
    os.remove(SCOPEE_CHECKPOINT)
    print("Judge checkpoint cleared -- starting fresh")
else:
    print("Existing judge checkpoint:", os.path.exists(SCOPEE_CHECKPOINT), "(will resume if present)")


Existing judge checkpoint: False (will resume if present)


In [ ]:
import importlib
import json as _j
import evaluator
importlib.reload(evaluator)
from evaluator import LLMJudge, build_report

judge = LLMJudge()
gpu_mem("after judge load")

# Read from the fixed checkpoint path rather than relying on `results_path`
# staying defined in memory -- if the runtime restarted between Phase A and
# Phase B, that variable would be gone. This file survives as long as the
# same runtime session is still alive (it's on local disk, not Drive), but
# a full disconnect/restart wipes it too -- re-run generation in that case.
GENERATION_CHECKPOINT = f"{EVAL_DIR}/generation_checkpoint.json"
generation_results = _j.loads(open(GENERATION_CHECKPOINT).read())

scored_results, scored_path = judge.run_judging(generation_results, out_path=SCOPEE_CHECKPOINT)


GPU [after judge load]: 4.41/15.64 GB  (free: 11.23 GB)
🔄 Loading judge model m42-health/Llama3-Med42-8B in 4-bit NF4...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/439 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/698 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ BERTScore computed for all rows.


## 📈 Step 13 — Build Report & Dashboard

In [ ]:
from evaluator import build_report
import json as _j

report = build_report(scored_results)

def dashboard(r):
    sep = "=" * 70
    nf  = r["not_found_count"]
    pct = int(nf / r["n_questions"] * 100) if r["n_questions"] else 0
    print(sep)
    print("  ONCRAG — EVALUATION DASHBOARD")
    print(f"  {r['n_questions']} questions  |  Not-found: {nf} ({pct}%)  |  "
          f"Avg agent iters: {r['avg_agent_iters']}  |  Avg latency: {r['avg_latency_sec']}s")
    print(sep)

    print("\n── RETRIEVAL (k=5) ──────────────────────────────────")
    for k, v in r["retrieval"].items():
        print(f"  {k:<10}: {v}")

    print("\n── LEXICAL ──────────────────────────────────────────")
    for k, v in r["lexical"].items():
        print(f"  {k:<10}: {v}")

    print(f"\n── SEMANTIC ─────────────────────────────────────────")
    print(f"  BERTScore F1                        : {r['bertscore_f1']}")
    print(f"  Faithfulness (judged by Llama3-Med42): {r['faithfulness']}  "
          f"✅ 0-1 scale, judged against the actual retrieved context "
          f"(HHEM has been removed — this replaces it)")

    print("\n── S.C.O.P.E.E (judged by Llama3-Med42) ─────────────")
    for k, v in r["scopee"].items():
        print(f"  {k:<14}: {v}")

    if r.get("by_category"):
        print("\n── BY CATEGORY ──────────────────────────────────────")
        for cat, stats in r["by_category"].items():
            print(f"  {cat:<14} n={stats['n']:<4} faithfulness={stats['avg_faithfulness']}  bertscore={stats['avg_bertscore_f1']}")

    if r.get("by_difficulty"):
        print("\n── BY DIFFICULTY ────────────────────────────────────")
        for diff, stats in r["by_difficulty"].items():
            print(f"  {diff:<14} n={stats['n']:<4} faithfulness={stats['avg_faithfulness']}  bertscore={stats['avg_bertscore_f1']}")

    print(sep)

dashboard(report)

with open(f"{EVAL_DIR}/report.json", "w") as f:
    _j.dump(report, f, indent=2)
print(f"\n✅ Report saved -> {EVAL_DIR}/report.json")


## 💬 Step 14 — Interactive Chat Demo
Reloads a fresh Mistral chain for live Q&A (run after Steps 11-13, or independently).
**Do not run this while the judge model from Step 12 is still loaded** — Mistral +
Med42 together will not fit on a T4. If `judge` still exists in this session, free it
first: `del judge; free_vram()`.

In [ ]:
from chain import build_chain

onc_chain = build_chain(chunks_json=CHUNKS_JSON, index_path=INDEX_PATH, top_k=5, verbose=True)

def chat(question):
    print(f"\n{'='*65}")
    print(f"Q: {question}")
    r = onc_chain.ask(question)
    print(f"\nA: {r['answer']}")
    print(f"Faithfulness (in-loop retry heuristic, NOT the judged metric): {r['faithfulness']:.2f}")
    print(f"Agent iters  : {r['agent_iters']}")
    print(f"Query type   : {r['query_type']}   Disease: {r['disease']}")
    print(f"Latency      : {r['latency_sec']}s")

chat("What are the first-line treatment options for HER2-positive breast cancer?")


## 💾 Step 15 — Download Results & Cache

In [ ]:
from google.colab import files as _f
import glob as _g

result_files = sorted(_g.glob(f"{EVAL_DIR}/*.json") + _g.glob(f"{EVAL_DIR}/*.txt"))
print(f"Downloading {len(result_files)} result files:")
for f in result_files:
    print(f"  {os.path.basename(f)}  ({os.path.getsize(f)/1e3:.0f} KB)")
    _f.download(f)

cache_files = [CHUNKS_JSON, INDEX_PATH + ".pkl", INDEX_PATH + ".faiss"]
for f in cache_files:
    if os.path.exists(f):
        print(f"Downloading cache: {os.path.basename(f)}  ({os.path.getsize(f)/1e6:.1f} MB)")
        _f.download(f)
    else:
        print(f"❌ Not found: {f}")


  generation_checkpoint.json  (951 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  report.json  (3 KB)


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 🧪 Step 16 (Optional/Experimental) — Disease-Penalty Retrieval Patch
Everything below is exploratory: it monkey-patches the retriever to also
**penalize** (not just boost) chunks tagged with a confirmed wrong disease, and
compares it against baseline using the judge's chunk-relevance metric.

Not required for the main evaluation — skip this section unless you're
specifically testing the disease-penalty idea. Needs `onc_chain` (Mistral) and
`judge` (Med42) both available; per Step 11's note, having both loaded at once
risks OOM on a T4 — keep an eye on `gpu_mem()`.

In [ ]:
import torch
print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"GPU memory reserved: {torch.cuda.memory_reserved()/1e9:.2f} GB")

GPU memory used: 10.11 GB
GPU memory reserved: 12.59 GB


In [ ]:
print('onc_chain' in globals())  # just checking, don't run build_chain regardless
print('scored_results' in globals())  # this is what actually matters

False
True


In [ ]:
import json
scored_results = json.loads(open(SCOPEE_CHECKPOINT).read())  # or your backup path
print(f"Loaded {len(scored_results)} scored results")

Loaded 200 scored results


In [ ]:
print("scored_results available:", 'scored_results' in globals())
print("evaluator module available:", 'evaluator' in globals())

scored_results available: True
evaluator module available: True


In [ ]:
query = "What are the major goals when treating carcinoma of the larynx?"
chunks = onc_chain.retriever.retrieve(query, k=5, primary_disease="larynx")
for c in chunks:
    print(round(c["score"], 4), c["disease"], "-", c["text"][:70])


In [ ]:
from evaluator import judged_retrieval_metrics

In [ ]:
import json, random
from query_analyzer import analyze  # OncRAGChain has no `.query_analyzer` attribute --
                                     # analyze() is the module-level LAQA function it
                                     # calls internally. Calling it directly here fixes
                                     # the AttributeError this cell used to swallow
                                     # silently (which meant primary_disease was always
                                     # None and the whole comparison never ran for real).

with open(QA_JSON) as f:
    qa_set = json.load(f)

random.seed(42)
subset = random.sample(qa_set, min(40, len(qa_set)))

def eval_retriever(label, use_penalty):
    precisions, mrrs, ndcgs, hits = [], [], [], []
    for item in subset:
        try:
            analysis = analyze(item["q"])
            primary_disease = analysis.primary_disease
        except Exception:
            primary_disease = None

        if use_penalty:
            chunks = onc_chain.retriever.retrieve(item["q"], k=5, primary_disease=primary_disease)
        else:
            # bypass patch: call original unpatched logic by temporarily zeroing the penalty effect
            chunks = onc_chain.retriever.retrieve(item["q"], k=5, primary_disease=primary_disease,
                                                    disease_penalty=1.0)  # 1.0 = no penalty = baseline

        m = judged_retrieval_metrics(judge, item["q"], chunks)
        if m.get("precision_5") is not None: precisions.append(m["precision_5"])
        if m.get("mrr") is not None: mrrs.append(m["mrr"])
        if m.get("ndcg_5") is not None: ndcgs.append(m["ndcg_5"])
        if m.get("hit_rate_5") is not None: hits.append(m["hit_rate_5"])

    avg = lambda lst: round(sum(lst)/len(lst), 4) if lst else None
    print(f"\n=== {label} (n={len(subset)}) ===")
    print("precision_5:", avg(precisions))
    print("mrr        :", avg(mrrs))
    print("ndcg_5     :", avg(ndcgs))
    print("hit_rate_5 :", avg(hits))

eval_retriever("BASELINE (no penalty)", use_penalty=False)
eval_retriever("WITH DISEASE PENALTY (0.5x)", use_penalty=True)


In [ ]:
# ── Lenient retrieval scoring (word-overlap threshold 0.20 -> 0.10) ──────────
# Disclose this threshold choice in your report's methodology section if you
# use these numbers -- this does NOT touch evaluator.py on disk, it patches
# the function in-memory for this session only, and recomputes instantly
# from your existing scored_results (no regeneration needed).
import evaluator
import math

def ref_overlap_retrieval_metrics_lenient(chunks, ref, k=5, min_overlap=0.10):
    ...

In [ ]:
import json
from evaluator import _retrieval_metrics
import numpy as np

with open(QA_JSON) as f:
    all_qa = json.load(f)

agg = {k: [] for k in ["precision_5", "recall_5", "mrr", "ndcg_5", "hit_rate_5"]}
for item in all_qa:
    chunks_r = retriever.retrieve(item["q"], k=5, use_rerank=True, alpha=0.5, fetch_k=20)
    m = _retrieval_metrics(chunks_r, item.get("relevant_chunk_ids"), gold_answer=item.get("a"))
    for key in agg:
        if m.get(key) is not None:
            agg[key].append(m[key])

print("Retrieval metrics with tuned config, on real gold labels, full 200 questions:")
for key, vals in agg.items():
    print(f"  {key}: {np.mean(vals):.4f}" if vals else f"  {key}: n/a")

🔨 Loading reranker: cross-encoder/ms-marco-MiniLM-L-6-v2


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Retrieval metrics with tuned config, on real gold labels, full 200 questions:
  precision_5: 0.6670
  recall_5: 0.9600
  mrr: 0.8527
  ndcg_5: 0.8667
  hit_rate_5: 0.9600
